# Updates in Week 3

Similarity Estimator reads from the hd5 file created by embeddingsGenerator, computes the similarity for each user id, keeps track of the top K by using a heap, and in the end writes the result into another hd5 file. 

The output hd5 files are as follows: one file for each similarity metric, the similarity metric file is indexed by the user key, where a list of the nearest k ids and distances are stored.

Documentation added to the classes: EmbeddingsGenerator, SimilarityEstimator, and SimilarityServer.

In attempt to summarize the confusion matrix produced by the different similarity metrics. The annotated dataset from https://github.com/PrincetonML/SIF/blob/master/data/sicktest was used as a source of truth, a sample was taken from the dataset, the scores from the sample were normalized from 0 to 1. The estimated similarities on the sample were also normalized from 0 to 1, and the L2 norm was calculated between the two vectors, this serves as a measurement of error. 
These errors were plotted for the different models in the following notebook:https://colab.research.google.com/drive/1eC4OFfCOkzjTEBdMCG07IRn7JE0r_A14?usp=sharing . The highest achieving metric was Universal sentence encoder with cosine similarity.
Another comparison was made between bert and albert models in the following notebook: https://colab.research.google.com/drive/1oXgWy4z6tZUecP4Z7vTPDETDPluPlp5E?usp=sharing


## Install Required Libraries

In [0]:
%%capture
# Install the latest Tensorflow version.
!pip3 install --upgrade tensorflow-gpu
# Install TF-Hub.
!pip3 install tensorflow==2.0
!pip3 install tensorflow-hub
!pip3 install bert-for-tf2
!pip3 install sentencepiece

## Import Required Libraries to Python

In [0]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import tensorflow_hub as hub
import tensorflow as tf 
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sentencepiece as spm
import tokenization
import time
import bert
import h5py
FullTokenizer = bert.bert_tokenization.FullTokenizer

from tensorflow.keras.models import Model
from sklearn.preprocessing import minmax_scale
from tabulate import tabulate

from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial import distance
from numpy import linalg as LA

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Mount Google Drive

In [0]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Helper Classes: BERT Model


In [0]:
from BertModel import BertModel

## Helper Classes: Albert Model

In [0]:
from AlbertModel import AlbertModel

## Helper Classes: Embeddings Generator

In [0]:
from EmbeddingsGenerator import EmbeddingsGenerator

## Similarity Calculator

In [0]:
from SimilarityEstimator import SimilarityEstimator

## Similarity Server

In [0]:
from SimilarityServer import SimilarityServer

## PlayGround

In [0]:
embeddingsGenerator = EmbeddingsGenerator(delete_models=False)

In [0]:
simcal = SimilarityEstimator()

In [0]:
similarity_server = SimilarityServer(embeddingsGenerator, simcal)
print(similarity_server.get_available_embedding_algorithms())
print(similarity_server.get_available_distance_metrics())

['elmo', 'use', 'word2vec', 'nnlm', 'bert', 'albert']
['cosine', 'minkowski', 'correlation', 'jensenshannon', 'cityblock', 'sqeuclidean', 'euclidean', 'chebyshev', 'canberra', 'braycurtis']


In [0]:
input_csv = 'minyset.csv'

In [0]:
similarity_server.create_similarity_index(input_csv, model_names=['elmo', 'use'])

Writing To  /content/drive/My Drive/NZBWW_elmo_embeddings.h5
elmo : Loading
elmo  Already Loaded
elmo  Already Loaded
Done Writing elmo embeddings
Writing To  /content/drive/My Drive/NZBWW_use_embeddings.h5
use : Loading
use  Already Loaded
use  Already Loaded
Done Writing use embeddings
Opening  /content/drive/My Drive/NZBWW_elmo_embeddings.h5

Starting elmo Chunk /chunk_0
Writing To /content/drive/My Drive/elmo_cosine_index.h5... Done Writing cosine data
Writing To /content/drive/My Drive/elmo_minkowski_index.h5... Done Writing minkowski data
Writing To /content/drive/My Drive/elmo_correlation_index.h5... Done Writing correlation data
Writing To /content/drive/My Drive/elmo_jensenshannon_index.h5... Done Writing jensenshannon data
Writing To /content/drive/My Drive/elmo_cityblock_index.h5... Done Writing cityblock data
Writing To /content/drive/My Drive/elmo_sqeuclidean_index.h5... Done Writing sqeuclidean data
Writing To /content/drive/My Drive/elmo_euclidean_index.h5... Done Writin

In [0]:
similarity_server.get_most_similar_K_to(4122, 5, 'use', 'cosine');

['/key_1', '/key_1103', '/key_1355', '/key_1486', '/key_15', '/key_1691', '/key_187', '/key_1888', '/key_22', '/key_2287', '/key_2665', '/key_28', '/key_2925', '/key_3', '/key_4', '/key_4122', '/key_51', '/key_52', '/key_53', '/key_54', '/key_55', '/key_56', '/key_57', '/key_58', '/key_59']
two young women with folded arms are looking at the screen
----  --------  ----------------------------------------------------------------------
   1  0.126156  two women with folded arms , who are young , are looking at the screen
2925  0.620487  two children are crouching under some metal bars
  57  0.635718  a woman in a purple top is walking a bicycle between two stone columns
   3  0.641305  two children are crouching under some bars made of metal
  56  0.668657  a woman is putting cosmetics on her eyelid
----  --------  ----------------------------------------------------------------------
